# Auditoria técnica — Seleção de FIIs

## tl;dr

- O ranking é reproduzível, mas insuficiente para seleção profissional: combina DY, P/VP e liquidez no mesmo universo para tipos economicamente diferentes.
- O banco real tem 316 FIIs, porém não possui as colunas de versionamento/frescor da migração 018 nem o snapshot de score da migração 020; portanto não permite validação point-in-time.
- A carteira padrão resultante tem 10 ativos, número efetivo de 8,38, mas fica 50% em papel e 50% em tijolo e cobre apenas três rótulos de segmento.
- A carteira de qualidade padrão reduz o universo a 20 fundos e resulta em 40% papel, 40% FoF e 20% tijolo, com somente dois rótulos amplos de segmento.
- Conclusão: metodologia útil para triagem exploratória, não confiável como recomendação autônoma de longo prazo sem revisão humana e dados adicionais.

## Context & Methods

Auditoria em 11/07/2026 do banco PostgreSQL configurado pela aplicação e da implementação em `views/fiis.py`, `data_pipeline/market/fii.py` e `core/market_read.py`. Os testes abaixo não alteram o banco.

### Key Assumptions

- Parâmetros de interface padrão foram usados para reconstruir as carteiras.
- P/VP efetivo = preço atual / VPA CVM quando ambos existem.
- A análise da carteira é descritiva e in-sample; não é validação preditiva.

In [1]:
import math

import pandas as pd
from sqlalchemy import text

from core import market_read as mr
from core.database import get_engine
from data_pipeline.market import fii as fz

engine = get_engine()
assert engine is not None, 'Banco não configurado'
pd.set_option('display.max_columns', 40)

## Data

### 1. Perfil de qualidade e migrações aplicadas

In [2]:
with engine.connect() as conn:
    fiis = pd.read_sql(text('SELECT * FROM market.fiis'), conn)
    monthly = pd.read_sql(text('SELECT * FROM market.fii_metrics_monthly'), conn)
    imoveis = pd.read_sql(text('SELECT * FROM market.fii_imoveis'), conn)
    raw_bars = pd.read_sql(text("""
      WITH latest AS (
        SELECT DISTINCT ON (r.ticker) r.ticker, r.endpoint, r.fetched_at, r.payload_json
        FROM market.brapi_raw_payloads r JOIN market.fiis f ON f.ticker=r.ticker
        WHERE r.endpoint IN ('quote_fii_full','quote') AND r.request_status='success'
        ORDER BY r.ticker, CASE WHEN r.endpoint='quote_fii_full' THEN 0 ELSE 1 END,
                 r.fetched_at DESC, r.id DESC
      )
      SELECT ticker, endpoint, fetched_at,
             jsonb_array_length(COALESCE(payload_json->'historicalDataPrice',
               payload_json->'results'->0->'historicalDataPrice','[]'::jsonb)) AS bars
      FROM latest
    """), conn)

critical = ['tipo','dy_12m','pvp','score','vpa','vacancia','num_imoveis']
profile = pd.DataFrame({
    'campo': critical,
    'ausentes_pct': [round(fiis[c].isna().mean()*100, 1) for c in critical],
})
print('FIIs:', len(fiis), '| atualização:', fiis.updated_at.max())
print('Colunas de metadados presentes:', [c for c in ['score_version','score_calculated_at','metrics_fetched_at'] if c in fiis])
print('Mensal:', len(monthly), 'linhas,', monthly.ticker.nunique(), 'tickers,', monthly.ref_month.min(), 'a', monthly.ref_month.max())
print('Snapshot de score presente:', 'score' in monthly.columns)
print('Imóveis:', len(imoveis), 'linhas e', imoveis.ticker.nunique(), 'fundos')
print('Payloads com menos de 6 barras:', int((raw_bars.bars < 6).sum()), 'de', len(raw_bars))
display(profile)

FIIs: 316 | atualização: 2026-07-03 22:17:26.696164+00:00
Colunas de metadados presentes: []
Mensal: 8513 linhas, 316 tickers, 2024-01-01 a 2026-05-01
Snapshot de score presente: False
Imóveis: 2310 linhas e 173 fundos
Payloads com menos de 6 barras: 279 de 316
      campo  ausentes_pct
       tipo           9.5
     dy_12m           8.9
        pvp           0.3
      score          24.7
        vpa           9.5
   vacancia          57.0
num_imoveis          45.3


## Results

### 2. Reconstrução do ranking e da carteira padrão

In [3]:
rows = []
for r in fiis.itertuples():
    rows.append({
        'ticker': r.ticker, 'price': r.price, 'dy_12m': r.dy_12m,
        'pvp': fz.pvp_efetivo(r.price, r.vpa, r.pvp),
        'liquidez_diaria': r.liquidez_diaria, 'tipo': r.tipo,
        'segmento': r.segmento_cvm or r.segmento,
    })
ranked = fz.rank_fiis(rows)
standard = fz.build_portfolio(ranked, n_max=10, max_weight=.20, max_tipo_frac=.50)
standard_df = pd.DataFrame(standard)
print('Elegíveis:', len(ranked), 'de', len(fiis), f'({len(ranked)/len(fiis):.1%})')
print('N efetivo:', fz.effective_n(dict(zip(standard_df.ticker, standard_df.peso))))
print('Peso por tipo:', standard_df.groupby('tipo').peso.sum().round(4).to_dict())
print('Rótulos de segmento:', standard_df.segmento.nunique())
display(standard_df)

Elegíveis: 125 de 316 (39.6%)
N efetivo: 8.38
Peso por tipo: {'papel': 0.5, 'tijolo': 0.5}
Rótulos de segmento: 3
ticker     peso  score   tipo       segmento   dy_12m      pvp
RBRY11 0.079518   85.1  papel Multicategoria 0.157295 0.873750
RZTR11 0.176953   82.0 tijolo         Outros 0.139129 0.883176
RZAK11 0.075313   80.6  papel Multicategoria 0.149528 0.924151
VGIR11 0.072790   77.9  papel Multicategoria 0.156602 0.997307
HGCR11 0.072603   77.7  papel         Outros 0.131971 0.919589
LASC11 0.161847   75.0 tijolo      Shoppings 0.155232 0.848908
TRXF11 0.161200   74.7 tijolo Multicategoria 0.129170 0.929803
RECR11 0.066810   71.5  papel Multicategoria 0.129315 0.933921
SAPI11 0.066716   71.4  papel Multicategoria 0.157910 0.908215
VRTA11 0.066249   70.9  papel         Outros 0.144989 0.839466


### 3. Reconstrução da carteira de qualidade com parâmetros padrão

In [4]:
q = mr.load_fii_quality().copy()
f = q[q.Tipo.isin(['tijolo','hibrido','papel','fof'])]
f = f[f.Liquidez_Diaria.fillna(0) >= 1_000_000]
f = f[f.DY_12m.between(.08, .20, inclusive='both')]
f = f[f['P/VP'].between(.55, 1.30, inclusive='both')]
f = f[f.Max_Drawdown.fillna(0) >= -.35]
f = f[f.Hist_Meses.fillna(0) >= 24]
f = f[f['P/VP'].fillna(9) < 1]
brick = f.Tipo.isin(['tijolo','hibrido'])
keep = ((~brick) | (f.N_Regioes.fillna(0) >= 2)) & ((~brick) | (f.Num_Imoveis.fillna(0) >= 8)) & ((~brick) | f.Multi_Setorial)
f = f[keep].copy()
def rk(s, higher=True):
    r = s.rank(pct=True).fillna(s.rank(pct=True).median() if s.notna().any() else .5)
    return r if higher else 1-r
div_brick = .4*(f.N_Regioes.fillna(0).clip(upper=5)/5) + .3*(f.Num_Imoveis.fillna(0).clip(upper=40)/40) + .3*f.Multi_Setorial.astype(float)
div = div_brick.where(f.Tipo.isin(['tijolo','hibrido']), .5)
growth = .6*rk(f.CAGR) + .4*rk(f.Max_Drawdown)
pvp_dist = f['P/VP'].map(lambda x: abs(math.log(x/.9)) if x and x > 0 else None)
f['Qualidade'] = 100*(.35*rk(f.DY_12m)+.25*growth+.25*div+.10*rk(f.Liquidez_Diaria)+.05*rk(pvp_dist, False))
quality_rows = [{'ticker':r.Ticker,'score':r.Qualidade,'tipo':r.Tipo,'liquidez_diaria':r.Liquidez_Diaria,'dy_12m':r.DY_12m,'pvp':r['P/VP'],'segmento':r.Segmento} for _,r in f.iterrows()]
quality = fz.build_portfolio(quality_rows, n_max=10, max_weight=.20, max_tipo_frac=.40, liq_min=1_000_000, min_por_tipo=1)
quality_df = pd.DataFrame(quality)
print('Elegíveis:', len(f), '| por tipo:', f.Tipo.value_counts().to_dict())
print('N efetivo:', fz.effective_n(dict(zip(quality_df.ticker, quality_df.peso))))
print('Peso por tipo:', quality_df.groupby('tipo').peso.sum().round(4).to_dict())
print('Rótulos de segmento:', quality_df.segmento.nunique())
display(quality_df)

Elegíveis: 20 | por tipo: {'papel': 13, 'fof': 4, 'tijolo': 3}
N efetivo: 8.31
Peso por tipo: {'fof': 0.4, 'papel': 0.4, 'tijolo': 0.2}
Rótulos de segmento: 2
ticker     peso  score   tipo       segmento   dy_12m    pvp
RZAK11 0.078728 72.750  papel Multicategoria 0.149528 0.9175
TRXF11 0.200000 66.500 tijolo Multicategoria 0.129170 0.9285
VGIR11 0.071153 65.750  papel Multicategoria 0.156602 0.9953
RBRY11 0.071153 65.750  papel Multicategoria 0.157295 0.8777
VGHF11 0.137725 57.500    fof         Outros 0.154362 0.7107
VCJR11 0.061684 57.000  papel         Outros 0.141713 0.7980
ITRI11 0.132335 55.250    fof         Outros 0.131830 0.8508
HABT11 0.059655 55.125  papel Multicategoria 0.165544 0.7499
JSAF11 0.129940 54.250    fof Multicategoria 0.136202 0.7885
HGCR11 0.057626 53.250  papel         Outros 0.131971 0.9189


### 4. Sensibilidade dos pesos do ranking

In [5]:
scenarios = {
 'default': None,
 'pesos_iguais': {'dy_12m':1/3,'pvp':1/3,'liquidez_diaria':1/3},
 'renda_baixa': {'dy_12m':.20,'pvp':.40,'liquidez_diaria':.40},
 'renda_alta': {'dy_12m':.65,'pvp':.20,'liquidez_diaria':.15},
}
top10 = {name:[r['ticker'] for r in fz.rank_fiis(rows, weights=w)[:10]] for name,w in scenarios.items()}
base = set(top10['default'])
sensitivity = pd.DataFrame([{'cenário':k,'sobreposição_top10':len(base & set(v)),'top10':', '.join(v)} for k,v in top10.items()])
display(sensitivity)

     cenário  sobreposição_top10                                                                          top10
     default                  10 RBRY11, RZTR11, RZAK11, VGIR11, HGCR11, EGAF11, LASC11, TRXF11, RECR11, SAPI11
pesos_iguais                   7 RZTR11, RBRY11, HGCR11, RZAK11, TRXF11, VGIR11, GARE11, RECR11, KNHF11, FGAA11
 renda_baixa                   5 RZTR11, HGCR11, TRXF11, RBRY11, GARE11, HGLG11, VISC11, RZAK11, KNHF11, XPLG11
  renda_alta                   7 RBRY11, VGIR11, RZAK11, LASC11, SAPI11, EGAF11, RZTR11, KORE11, HABT11, HSAF11


## Takeaways

1. Aplicar as migrações 018 e 020 e iniciar snapshots mensais antes de qualquer alegação de eficácia histórica.
2. Separar modelos por tipo: tijolo, papel, FoF e híbrido não devem compartilhar os mesmos benchmarks de DY e P/VP.
3. Substituir rótulos amplos por exposições econômicas: setor, gestor, locatário/devedor, indexador, duration, concentração e alavancagem.
4. Tratar a carteira atual como lista de diligência, não como recomendação automática.
5. Validar fora da amostra com rebalanceamentos point-in-time, custos, emissões, liquidez e comparação com IFIX sem viés de sobrevivência.